<a href="https://colab.research.google.com/github/everestso/everestso/blob/main/LLM_3b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GitHub Hosted MCP Server (Teacher-friendly Colab Demo)

This notebook connects to GitHub’s hosted MCP server over HTTP/SSE (no local server process).
We authenticate using a GitHub Personal Access Token (PAT) stored only in Colab runtime memory.


In [1]:
!pip -q install mcp httpx httpx-sse


In [2]:
import os
from google.colab import userdata

# Make sure your key is set:
os.environ["GITHUB_PAT"] = userdata.get("GITHUB_PAT")
GITHUB_PAT = os.environ["GITHUB_PAT"]

headers = {
    "Authorization": f"Bearer {GITHUB_PAT}"
}

In [3]:
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

GITHUB_MCP_URL = "https://api.githubcopilot.com/mcp/"

async def list_tools_streamable_http():
    # Put auth on the HTTP client (NOT as a streamable_http_client kwarg)
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {GITHUB_PAT}"},
        timeout=30.0,
    ) as http_client:
        async with streamable_http_client(GITHUB_MCP_URL, http_client=http_client) as (
            read_stream,
            write_stream,
            get_session_id,  # third return value; handy for debugging
        ):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                tools_response = await session.list_tools()
                tool_names = [t.name for t in tools_response.tools]

                print(f"Connected via Streamable HTTP. Session: {get_session_id()}")
                print(f"Found {len(tool_names)} tool(s). First 20:")
                for name in tool_names[:20]:
                    print(" -", name)

await list_tools_streamable_http()


Connected via Streamable HTTP. Session: 7b296d09-7a2f-412a-97aa-d0e1b51f88d1
Found 40 tool(s). First 20:
 - add_comment_to_pending_review
 - add_issue_comment
 - assign_copilot_to_issue
 - create_branch
 - create_or_update_file
 - create_pull_request
 - create_repository
 - delete_file
 - fork_repository
 - get_commit
 - get_file_contents
 - get_label
 - get_latest_release
 - get_me
 - get_release_by_tag
 - get_tag
 - get_team_members
 - get_teams
 - issue_read
 - issue_write


In [4]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
import httpx

GITHUB_MCP_URL = "https://api.githubcopilot.com/mcp/"

async def get_readme_via_github_mcp(owner: str, repo: str, path: str = "README.md"):
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {GITHUB_PAT}"},
        timeout=30.0,
    ) as http_client:
        async with streamable_http_client(GITHUB_MCP_URL, http_client=http_client) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                result = await session.call_tool(
                    "get_file_contents",
                    {
                        "owner": owner,
                        "repo": repo,
                        "path": path,
                    }
                )
                return result

# Call it
result = await get_readme_via_github_mcp("psf", "requests")
print(result)


meta=None content=[TextContent(type='text', text='successfully downloaded text file (SHA: 74adab80dd74ecbb38db4b34439dbbdd00be1736)', annotations=None, meta=None), EmbeddedResource(type='resource', resource=TextResourceContents(uri=AnyUrl('repo://psf/requests/70298332899f25826e35e42f8d83425124f755a5/contents/README.md'), mimeType='text/plain; charset=utf-8', meta=None, text='# Requests\n\n**Requests** is a simple, yet elegant, HTTP library.\n\n```python\n>>> import requests\n>>> r = requests.get(\'https://httpbin.org/basic-auth/user/pass\', auth=(\'user\', \'pass\'))\n>>> r.status_code\n200\n>>> r.headers[\'content-type\']\n\'application/json; charset=utf8\'\n>>> r.encoding\n\'utf-8\'\n>>> r.text\n\'{"authenticated": true, ...\'\n>>> r.json()\n{\'authenticated\': True, ...}\n```\n\nRequests allows you to send HTTP/1.1 requests extremely easily. There’s no need to manually add query strings to your URLs, or to form-encode your `PUT` & `POST` data — but nowadays, just use the `json` meth